In [1]:
from ta.volatility import AverageTrueRange
from ta.momentum import RSIIndicator
from ta.trend import EMAIndicator, SMAIndicator

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

from backtesting import Strategy
from backtesting import Backtest


from datetime import timedelta
import Util
import importlib
importlib.reload(Util)

c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

<module 'Util' from 'c:\\Users\\kyanh\\Documents\\Backtest\\OpenRangeBreakout\\utils\\Util.py'>

In [2]:
pair = 'GBPJPY'

df_d1_raw = pd.read_csv(f"../Trading Data/{pair}_D1.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_h4_raw = pd.read_csv(f"../Trading Data/{pair}_H4.csv",sep=',', parse_dates=['Date'], index_col='Date')
df_m15_raw = pd.read_csv(f"../Trading Data/{pair}_M15.csv",sep=',', parse_dates=['Date'], index_col='Date')

print(df_d1_raw)
print(df_h4_raw)
print(df_m15_raw)

               Open     High      Low    Close  Tickvol
Date                                                   
2010-09-10  129.605  130.065  128.995  129.255   172229
2010-09-13  129.565  130.115  128.780  129.090   215859
2010-09-14  129.105  129.430  127.640  129.015   226170
2010-09-15  129.015  134.070  128.555  133.975   252317
2010-09-16  133.975  134.260  132.970  134.150   222443
...             ...      ...      ...      ...      ...
2026-09-18  208.323  211.276  208.233  210.095   364707
2026-09-21  210.296  210.762  209.676  210.321   243731
2026-09-22  210.321  210.892  209.513  210.080   333200
2026-09-23  210.080  210.176  209.422  209.564   273936
2026-09-24  209.566  210.105  208.768  209.848   199988

[4183 rows x 5 columns]
                        Open     High      Low    Close  Tickvol
Date                                                            
2010-09-10 04:00:00  129.605  129.630  128.995  129.385    25539
2010-09-10 08:00:00  129.380  129.985  129.275  129.

In [3]:
structure_d1_window = 2
structure_m15_window = 2

date_format = '%Y-%m-%d %H:%M'
# start_date = '2023-1-1 00:00'
# end_date = '2024-3-29 00:00'
start_date = datetime.strptime('2022-1-1 00:00', date_format)
end_date = datetime.strptime('2022-12-30 00:00', date_format)
# start_date = '2022-01-1 00:00'
# end_date = '2026-9-16 00:00'

timezone = "Europe/London"

# nombre de jours pour les high.low
define_key_level_window = 90

gap_look_ahead = 6

# nombre de candle au dessus/ en desous de key level après breakout
breakout_candles = 9

# nombre de jours à chercher key level
key_level_lookback_candle = 100

# vérifier si 30 candle dans ke passé pour valider un valid breakout,
# un valid breakout doit avoir {breakout_candles} candles au dessus du resistant level
breakout_candles_lookback = 30

start_trade_hour = 9
stop_trade_hour = 19

atr_multiplier = 0.7
min_touched = 3

In [ ]:
# process D1 data
df_d1 = df_d1_raw.loc[start_date - timedelta(days=30) : end_date]

df_d1 = Util.implement_market_structure_states_fixed_pivots(df_d1, n=structure_d1_window)
df_d1["state_changed"] = df_d1["market_state"] != df_d1[
    "market_state"
].shift(1)
for state in df_d1[df_d1["state_changed"]][["Close", "market_state"]].iterrows():
    print(state)
df_d1 = df_d1.loc[start_date : end_date]

(Timestamp('2021-12-02 00:00:00'), Close           150.524
market_state        NaN
Name: 2021-12-02 00:00:00, dtype: object)
(Timestamp('2021-12-03 00:00:00'), Close           149.229
market_state        NaN
Name: 2021-12-03 00:00:00, dtype: object)
(Timestamp('2021-12-06 00:00:00'), Close           150.45
market_state       NaN
Name: 2021-12-06 00:00:00, dtype: object)
(Timestamp('2021-12-07 00:00:00'), Close           150.367
market_state        NaN
Name: 2021-12-07 00:00:00, dtype: object)
(Timestamp('2021-12-08 00:00:00'), Close           150.175
market_state        NaN
Name: 2021-12-08 00:00:00, dtype: object)
(Timestamp('2021-12-09 00:00:00'), Close           149.973
market_state        NaN
Name: 2021-12-09 00:00:00, dtype: object)
(Timestamp('2021-12-10 00:00:00'), Close           150.445
market_state        NaN
Name: 2021-12-10 00:00:00, dtype: object)
(Timestamp('2021-12-13 00:00:00'), Close           150.056
market_state        NaN
Name: 2021-12-13 00:00:00, dtype: object)
(T

In [ ]:
# process h4 data
df_h4 = df_h4_raw.loc[start_date - timedelta(days=20) : end_date]
df_h4 = Util.implement_market_structure_states_fixed_pivots(df_h4, 5)
df_h4 = df_h4.loc[start_date : end_date]

h4_d1_direction_list = []
for i in range(len(df_d1)):
    row = df_d1.iloc[i]
    start_date_daily = row.name
    end_date_daily = start_date_daily + timedelta(hours=23, minutes=59)
    df_by_day = df_h4.loc[f'{start_date_daily}' : f'{end_date_daily}']

    for index2, row2 in df_by_day.iterrows():
        h4_d1_direction_list.append(df_d1.iloc[i-1]['market_state'])

df_h4['market_state_d1'] = h4_d1_direction_list

indicator_atr_14 = AverageTrueRange(df_h4['High'], df_h4['Low'], df_h4['Close'], window=14)
df_h4['ATR'] = indicator_atr_14._atr

df_h4

,Open,High,Low,Close,Tickvol,is_swing_high,is_swing_low,market_state,market_state_d1,ATR
Date,,,,,,,,,,
2022-01-03 00:00:00,155.676,155.916,155.642,155.668,16493,False,False,BULLISH,BEARISH,0.000000
2022-01-03 04:00:00,155.667,155.776,155.562,155.762,14930,False,False,BULLISH,BEARISH,0.000000
2022-01-03 08:00:00,155.765,155.868,155.482,155.484,20302,False,False,BULLISH,BEARISH,0.000000
2022-01-03 12:00:00,155.486,155.596,155.121,155.196,28248,False,False,BULLISH,BEARISH,0.000000
2022-01-03 16:00:00,155.199,155.459,154.883,155.381,24726,False,True,BULLISH,BEARISH,0.000000
...,...,...,...,...,...,...,...,...,...,...
2022-12-29 08:00:00,160.820,161.165,160.746,161.037,42059,False,False,BEARISH,BEARISH,0.610855
2022-12-29 12:00:00,161.040,161.051,160.134,160.209,63837,False,False,BEARISH,BEARISH,0.632723
2022-12-29 16:00:00,160.210,160.839,160.045,160.510,44256,False,False,BEARISH,BEARISH,0.644243


In [6]:
supports_list = Util.define_support_level_with_atr(df_h4, define_key_level_window, atr_multiplier=atr_multiplier, min_touched=min_touched)

supports_list

[[(Timestamp('2022-02-15 08:00:00'),
   Open               155.989
   High               156.824
   Low                155.962
   Close              156.701
   Tickvol              55067
   is_swing_high         True
   is_swing_low         False
   market_state       BEARISH
   market_state_d1    BULLISH
   ATR                0.53696
   Name: 2022-02-15 08:00:00, dtype: object),
  156.726],
 [(Timestamp('2022-02-16 08:00:00'),
   Open                156.731
   High                157.111
   Low                 156.688
   Close               156.971
   Tickvol               42609
   is_swing_high          True
   is_swing_low          False
   market_state        BULLISH
   market_state_d1     BULLISH
   ATR                0.465813
   Name: 2022-02-16 08:00:00, dtype: object),
  156.93],
 [(Timestamp('2022-02-23 08:00:00'),
   Open                156.387
   High                 156.77
   Low                 156.281
   Close               156.502
   Tickvol               42696
   is_swi

In [7]:
df_h4['support_point'] = df_h4.apply(lambda row: Util.support_pos(supports_list, row), axis=1)
df_h4['is_support'] = df_h4.apply(lambda row: Util.is_support(supports_list, row), axis=1)

In [8]:
# df_h4['valid_breakout_support_levels'] = Util.is_candles_above_support_level_list(df_h4, breakout_candles, key_level_lookback_candle)
df_h4['valid_breakout_support_levels'] = Util.is_candles_above_support_level_list(df_h4, breakout_candles, key_level_lookback_candle)
df_h4['valid_breakout_support_levels']


Date
2022-01-03 00:00:00    [167.323, 168.961]
2022-01-03 04:00:00    [167.323, 168.961]
2022-01-03 08:00:00    [167.323, 168.961]
2022-01-03 12:00:00    [167.323, 168.961]
2022-01-03 16:00:00    [167.323, 168.961]
                              ...        
2022-12-29 08:00:00                    []
2022-12-29 12:00:00                    []
2022-12-29 16:00:00                    []
2022-12-29 20:00:00                    []
2022-12-30 00:00:00                    []
Name: valid_breakout_support_levels, Length: 1555, dtype: object

In [ ]:
# process m15 data
df_m15 = df_m15_raw.loc[start_date - timedelta(hours=3): end_date]
df_m15 = Util.implement_market_structure_states_fixed_pivots(df_m15, n=structure_m15_window)

m15_daily_direction_list = []

m15_h4_resistant_level_list = []
m15_is_resistants_list = []
m15_is_below_resistant_list = []

m15_h4_support_level_list = []
m15_is_supports_list = []
m15_is_above_support_list = []

index_list = []

df_m15 = df_m15.loc[start_date : end_date]
for i in range(len(df_h4)):
    row = df_h4.iloc[i]
    start_date_h4 = row.name
    end_date_h4 = start_date_h4 + timedelta(hours=3, minutes=59)
    df_by_day = df_m15.loc[f'{start_date_h4}' : f'{end_date_h4}']

    for index2, row2 in df_by_day.iterrows():
        # affect current market states of H4 candle because the result is market state of previous D1 candle
        m15_daily_direction_list.append(df_h4.iloc[i]['market_state_d1'])

        m15_h4_support_level_list.append(df_h4.iloc[i]['support_point'])
        m15_is_supports_list.append(df_h4.iloc[i]['is_support'])
        m15_is_above_support_list.append(df_h4.iloc[i]['valid_breakout_support_levels'])

        # ajust to daylight saving time
        dst = pd.Timestamp(index2, tz=timezone).dst()
        if dst == timedelta(0):
            index2 = index2 - timedelta(hours=1)
            index_list.append(index2)
        else:
            index_list.append(index2)

df_m15['market_state_d1'] = m15_daily_direction_list

df_m15['support_point'] = m15_h4_support_level_list
df_m15['is_support'] = m15_is_supports_list
df_m15['valid_breakout_support_levels'] = m15_is_above_support_list

df_m15.index = index_list

df_m15

,Open,High,Low,Close,Tickvol,is_swing_high,is_swing_low,market_state,market_state_d1,support_point,is_support,valid_breakout_support_levels
2022-01-02 23:00:00,155.676,155.776,155.676,155.731,1032,True,False,NaN,BEARISH,NaN,False,"[167.323, 168.961]"
2022-01-02 23:15:00,155.731,155.754,155.707,155.715,412,False,False,NaN,BEARISH,NaN,False,"[167.323, 168.961]"
2022-01-02 23:30:00,155.715,155.731,155.672,155.680,577,False,False,NaN,BEARISH,NaN,False,"[167.323, 168.961]"
2022-01-02 23:45:00,155.680,155.704,155.669,155.688,351,False,False,NaN,BEARISH,NaN,False,"[167.323, 168.961]"
2022-01-03 00:00:00,155.684,155.710,155.664,155.674,517,False,True,NaN,BEARISH,NaN,False,"[167.323, 168.961]"
...,...,...,...,...,...,...,...,...,...,...,...,...
2022-12-29 22:00:00,160.236,160.299,160.231,160.236,109,False,False,BEARISH,BEARISH,NaN,False,[]
2022-12-29 22:15:00,160.238,160.298,160.104,160.140,78,False,True,BEARISH,BEARISH,NaN,False,[]
2022-12-29 22:30:00,160.141,160.322,160.141,160.283,3267,False,False,BEARISH,BEARISH,NaN,False,[]
2022-12-29 22:45:00,160.285,160.307,160.203,160.273,4788,False,False,BEARISH,BEARISH,NaN,False,[]


In [10]:
print(Util.get_support_list(df_m15, 4193, key_level_lookback_candle, gap_look_ahead, Util.TimeFrame.M15))

support_point
156.726    156.726
156.737    156.737
156.930    156.930
Name: support_point, dtype: float64


In [11]:
# 2025-12-15 15:45:00 23438
print(Util.get_valid_breakout_support_level(df_m15, 4193, breakout_candles_lookback))

[]


In [12]:
def is_existed(list, value):
    for i in range(len(list)):
        if list[i] == value:
            return True
    return False

In [13]:
# using m15 market structure for entry
def apply_total_signal(df, key_level_lookback_candles = 120, gap=1, breakout_candles_lookback = 25, start_trade_hour=8, stop_trade_hour=19):
    # Initialize the 'TotalSignal' column
    df_signal_list = []
    valid_support_levels_list = []
    current_day = None

    # long variables
    is_key_level_touched = False
    is_above_support_levels = False

    high_tf_condition_long_satisfied = False
    # high_tf_condition_long_limit = 0
    # high_tf_condition_long_duration = 90
    high_tf_condition_long_candle = None
    m15_market_state_bullish = False

    # we want market structure to switch from bearish to bullish when htf conditions are met
    # if market structure is already bullish we need to wait
    previous_market_structure = None

    supports_levels_touched = []

    for i in range(0, len(df)):
        current_bar_date = df.index[i]
        signal = 0
        valid_support_levels = []

        if current_day != current_bar_date.date() or current_bar_date.time().hour >= stop_trade_hour:
            current_day = current_bar_date.date()
            # long
            is_key_level_touched = False
            valid_breakout_support_levels = []
            is_above_support_levels = False
            high_tf_condition_long_satisfied = False
            supports_levels_touched = []

        # if high_tf_condition_long_limit <= i:
        #     high_tf_condition_long_satisfied = False
        #     supports_levels_touched = []

        # h4 support level conditions
        supports_levels = Util.get_support_list(df, i, key_level_lookback_candles, gap, Util.TimeFrame.M15)
        for level in supports_levels:
            if df['Low'].iloc[i-1] < level:
                is_key_level_touched = True
                if not is_existed(supports_levels_touched, level):
                    supports_levels_touched.append(level)
        
        # long
        # d1 trending conditions
        c_long_daily_direction = df['market_state_d1'].iloc[i-1] == 'BULLISH'
        if c_long_daily_direction:
            c_m15_ms_bullish = df['market_state'].iloc[i] == 'BULLISH'

            # breakout_candles_above candle needs to be above support level for the breakout to be valid
            valid_breakout_support_levels = Util.get_valid_breakout_support_level(df, i, breakout_candles_lookback)

            for valid_level in valid_breakout_support_levels:
                for level_touched in supports_levels_touched:
                    if valid_level == level_touched:
                        valid_support_levels.append(valid_level)
                        is_above_support_levels = True

            # Combine conditions
            if (is_key_level_touched and
                not high_tf_condition_long_satisfied):
                high_tf_condition_long_satisfied = True
                # high_tf_condition_long_limit = i + high_tf_condition_long_duration
                m15_market_state_bullish = False
                high_tf_condition_long_candle = current_bar_date
                previous_market_structure = df['market_state'].iloc[i]
                print(f"high_tf_condition_long_satisfied {current_bar_date} {i}")

            if (high_tf_condition_long_satisfied and
                c_m15_ms_bullish and
                high_tf_condition_long_candle < current_bar_date and
                previous_market_structure == 'BEARISH'):
                m15_market_state_bullish = True
                print(f"m15_market_state_bullish {current_bar_date} {i}")

            # if i == 4193:
            #     print(supports_levels)
            #     print(supports_levels_touched)
            #     print(valid_breakout_support_levels)
            #     print(valid_support_levels)
            #     print(is_above_support_levels)

            c_time_trade = start_trade_hour <= current_bar_date.hour and current_bar_date.hour <= stop_trade_hour
            if c_time_trade:
                if (high_tf_condition_long_satisfied and
                    m15_market_state_bullish and
                    is_above_support_levels):
                    print(f"signal {current_bar_date} {i}")
                    signal = 2

        if previous_market_structure != df['market_state'].iloc[i]:
            if df['market_state'].iloc[i] == 'BEARISH':
                m15_market_state_bullish = False
            previous_market_structure = df['market_state'].iloc[i]    

        df_signal_list.append(signal)
        valid_support_levels_list.append(valid_support_levels)

    return df_signal_list, valid_support_levels_list

df_m15['TotalSignal'], df_m15['valid_support_levels'] = apply_total_signal(df=df_m15, key_level_lookback_candles=key_level_lookback_candle, gap=gap_look_ahead, 
                                                                           breakout_candles_lookback=breakout_candles_lookback, start_trade_hour=start_trade_hour, stop_trade_hour=stop_trade_hour)

high_tf_condition_long_satisfied 2022-02-15 08:45:00 2991
high_tf_condition_long_satisfied 2022-02-15 19:00:00 3032
high_tf_condition_long_satisfied 2022-02-15 19:15:00 3033
high_tf_condition_long_satisfied 2022-02-15 19:30:00 3034
high_tf_condition_long_satisfied 2022-02-15 19:45:00 3035
high_tf_condition_long_satisfied 2022-02-15 20:00:00 3036
high_tf_condition_long_satisfied 2022-02-15 20:15:00 3037
high_tf_condition_long_satisfied 2022-02-15 20:30:00 3038
high_tf_condition_long_satisfied 2022-02-15 20:45:00 3039
high_tf_condition_long_satisfied 2022-02-15 21:00:00 3040
high_tf_condition_long_satisfied 2022-02-15 21:15:00 3041
high_tf_condition_long_satisfied 2022-02-15 21:30:00 3042
high_tf_condition_long_satisfied 2022-02-15 21:45:00 3043
high_tf_condition_long_satisfied 2022-02-15 22:00:00 3044
high_tf_condition_long_satisfied 2022-02-15 22:15:00 3045
high_tf_condition_long_satisfied 2022-02-15 22:30:00 3046
high_tf_condition_long_satisfied 2022-02-15 22:45:00 3047
high_tf_condit

In [14]:
len(df_m15[df_m15.TotalSignal != 0])

126

In [15]:
df_m15['pointpos'] = df_m15.apply(lambda row: Util.pointpos(row), axis=1)

In [16]:
# add some sl tp management
def SIGNAL():
    return dfopt.TotalSignal

class MyStrat(Strategy):
    mysize = 0.5
    slcoef = 2
    TPcoef = 2
    risk_percent = 0.01
    last_swing_low = None
    last_swing_high = None
    previous_swing_low = None
    previous_swing_high = None
    key_level_gap = 0.2
    last_trade_index = 0

    stoploss_info_list = []
    valid_support_level = None

    setup_duration_gap = 60
    stoploss_window = 150
    
    def init(self):
        self.current_day = None
        self.order_placed_time = None
        self.setup_duration = None

        self.stoploss_info_list = []
        self.valid_support_level = None

        super().init()
        self.signal1 = self.I(SIGNAL)

    def reset_order(self):
        # delete placed order at the end of the day if not executed
        if self.order_placed_time:
            for order in self.orders:
                order_executed= False
                for trade in self.trades:
                    if trade.tag == order.tag:
                        order_executed = True

                if order_executed == False:
                    order.cancel()

        self.order_placed_time = None
        self.setup_duration = None
        self.valid_support_level = None

    def get_lowest_low(self):
        return self.data.Low[len(self.data.Low) - self.stoploss_window : len(self.data.Low)].min()

    def is_entry_in_valid_zone(self, valid_support_list):
        return self.data.Close[-1] > min(valid_support_list) - self.key_level_gap

    def add_stoploss_info_list(self, info):
        is_added = False
        for stoploss_info in self.stoploss_info_list:
            if stoploss_info[0] == info[0]:
                is_added = True
                stoploss_info[1] = info[1]

        if is_added == False:
            self.stoploss_info_list.append([info[0], info[1]])

    def get_stoploss_info(self, support_level):
        for stoploss_info in self.stoploss_info_list:
            if stoploss_info[0] == support_level:
                return stoploss_info

        return None

    def dynamic_stoploss(self):
        # delete placed order at the end of the day if not executed
        if self.order_placed_time:
            for order in self.orders:
                order_executed= False
                for trade in self.trades:
                    if trade.tag == order.tag:
                        order_executed = True
                        if order.stop is not None:
                            self.add_stoploss_info_list([self.valid_support_level, order.stop])

                if order_executed == False:
                    order.cancel()
                    if order.is_long:
                        sl = order.stop - self.get_lowest_low()
                        tp = order.stop + self.TPcoef * sl
                        position_size = int((self.risk_percent * self.equity) / sl)
                        order = self.buy(sl = self.get_lowest_low(), tp = tp, size=position_size, stop=order.stop)
        
    def get_tag(self):
        return f"{self.order_placed_time}"

    def wait_3_candles_before_new_trade(self):
            return self.last_trade_index + 3
    
    def next(self):
        super().next()
        index = len(self.data.index)

        # when an order is placed, it will be valable for 12h
        if self.setup_duration is not None and index >= self.setup_duration:
            self.reset_order()

        # place sl to the lowest/highest
        self.dynamic_stoploss()

        if len(self.trades) > 0:
            self.last_trade_index = index

        if len(self.orders)==0 and index >= self.wait_3_candles_before_new_trade():
            if self.signal1[-1]==2:
                valid_support_list = Util.get_valid_support_level(dfopt, index, breakout_candles_lookback)
                if len(valid_support_list) > 0:

                    valid_support_level = min(valid_support_list)
                    is_entry_in_valid_zone = self.is_entry_in_valid_zone(valid_support_list)
                    planned_entry_price = 0
                    sl = 0
                    
                    if is_entry_in_valid_zone: 
                        planned_entry_price = self.data.Close[-1]
                        sl = abs(planned_entry_price - self.get_lowest_low())
                    else :
                        planned_entry_price = valid_support_level - self.key_level_gap
                        sl = abs(planned_entry_price - self.get_lowest_low())

                    sl1 = planned_entry_price - sl
                    stoploss_info = self.get_stoploss_info(valid_support_level)
                    # print(f"{self.data.index[-1]} {stoploss_info} {valid_support_level}")
                    if stoploss_info is None or sl1 < stoploss_info[1]:
                        tp = self.TPcoef * sl
                        position_size = int((self.risk_percent * self.equity) / sl)
                        tp1 = planned_entry_price + tp

                        self.valid_support_level = valid_support_level
                        self.setup_duration = len(self.data.index) + self.setup_duration_gap
                        self.order_placed_time = self.data.index[-1]

                        order = None
                        if is_entry_in_valid_zone: 
                            order = self.buy(sl=sl1, tp=tp1, size=position_size, tag=self.get_tag())
                            print(f"short {self.data.index[-1]} sl = {sl1} & sgnal = {self.signal1[-1]} & valid_resistant_level = {valid_support_level} & entry_price = {planned_entry_price}")
                        else:
                            order = self.buy(sl=sl1, tp=tp1, size=position_size, stop=planned_entry_price, tag=self.get_tag())
                            print(f"short {self.data.index[-1]} sl = {sl1} & sgnal = {self.signal1[-1]} & valid_resistant_level = {valid_support_level} & planned_entry_price = {planned_entry_price}")

In [18]:
start_date_chart = '2022-1-1 00:00'
end_date_chart = '2022-12-30 00:00'
dfopt = df_m15.loc[start_date_chart : end_date_chart]
# dfopt = df_m15

# bt = Backtest(dfopt, MyStrat, cash=10000, margin=0.02, commission=0.0002)
bt = Backtest(dfopt, MyStrat, cash=10000, margin=0.02) #0.0002
stats = bt.run()

print(stats)

bt.plot()

short 2022-06-16 13:30:00 sl = 159.983 & sgnal = 2 & valid_resistant_level = 161.044 & entry_price = 162.355
short 2022-09-21 09:30:00 sl = 162.304 & sgnal = 2 & valid_resistant_level = 162.526 & entry_price = 163.065
short 2022-11-01 13:45:00 sl = 169.654 & sgnal = 2 & valid_resistant_level = 169.78 & entry_price = 170.26
short 2022-11-01 17:45:00 sl = 169.469 & sgnal = 2 & valid_resistant_level = 169.78 & entry_price = 170.236
short 2022-11-04 14:00:00 sl = 165.08 & sgnal = 2 & valid_resistant_level = 169.78 & planned_entry_price = 169.58
short 2022-11-07 09:00:00 sl = 165.08 & sgnal = 2 & valid_resistant_level = 169.78 & planned_entry_price = 169.58
short 2022-11-08 15:15:00 sl = 166.56 & sgnal = 2 & valid_resistant_level = 169.78 & planned_entry_price = 169.58
Start                     2022-01-02 23:00:00
End                       2022-12-29 23:00:00
Duration                    361 days 00:00:00
Exposure Time [%]                     1.45584
Equity Final [$]                     9897

c:\Users\kyanh\AppData\Local\Programs\Python\Python313-arm64\Lib\site-packages\backtesting\_plotting.py:141: UserWarning: Data contains too many candlesticks to plot; downsampling to '1h'. See `Backtest.plot(resample=...)`
  warnings.warn(f"Data contains too many candlesticks to plot; downsampling to {freq!r}. "


GridPlot(id='p1308', ...)

In [ ]:
start_date_chart = '2026-1-1 00:00'
end_date_chart = '2026-2-28 00:00'
dfpl = df_m15.loc[start_date_chart : end_date_chart]#.set_index("Gmt time")

# Create a plot with 2 rows
fig = make_subplots(rows=1, cols=1)

# Add candlestick plot on the first row
fig.add_trace(go.Candlestick(x=dfpl.index,
                             open=dfpl['Open'],
                             high=dfpl['High'],
                             low=dfpl['Low'],
                             close=dfpl['Close']),
              row=1, col=1)

#Add markers for trade entry points on the same subplot
# fig.add_trace(go.Scatter(x=dfpl.index, y=dfpl['support_point'], mode="markers",
#                          marker=dict(size=8, color="green"),
#                          name="support point"),
#               row=1, col=1)

# Add markers for entry point m15
fig.add_trace(go.Scatter(x=dfpl.index, y=dfpl['pointpos'], mode="markers",
                         marker=dict(size=12, color="MediumPurple"),
                         name="entry"),
              row=1, col=1)

# # Add markers for market structure daily = bullish
# fig.add_trace(go.Scatter(x=dfpl.index, y=dfpl['mk_bullish_pos'], mode="markers",
#                          marker=dict(size=8, color="green"),
#                          name="bullish"),
#               row=1, col=1)

# # Add markers for market structure daily = bearish
# fig.add_trace(go.Scatter(x=dfpl.index, y=dfpl['mk_bearish_pos'], mode="markers",
#                          marker=dict(size=8, color="red"),
#                          name="bearish"),
#               row=1, col=1)

fig.update_layout(width=1500, height=800, sliders=[])
fig.show()

In [19]:
start_date_chart = '2022-1-1 00:00'
end_date_chart = '2022-12-30 00:00'
dfpl = df_h4.loc[start_date_chart : end_date_chart]#.set_index("Gmt time")

# Create a plot with 2 rows
fig = make_subplots(rows=1, cols=1)

# Add candlestick plot on the first row
fig.add_trace(go.Candlestick(x=dfpl.index,
                             open=dfpl['Open'],
                             high=dfpl['High'],
                             low=dfpl['Low'],
                             close=dfpl['Close']),
              row=1, col=1)

#Add markers for trade entry points on the same subplot
fig.add_trace(go.Scatter(x=dfpl.index, y=dfpl['support_point'], mode="markers",
                         marker=dict(size=8, color="green"),
                         name="support point"),
              row=1, col=1)

fig.update_layout(width=1200, height=800, sliders=[])
fig.show()